Connected to sampl (Python 3.10.4)

In [ ]:
import os
import pandas as pd # pandas library
import numpy as np # numpy
import seaborn as sns
import matplotlib.pyplot as plt
from plot_functions.get_data_dir import (get_data_dir, get_figure_dir)
from plot_functions.get_bout_features import (get_aligned_bouts_wIBI, get_bout_features, extract_bout_features_v5)
from plot_functions.get_bout_kinetics import get_kinetics
from plot_functions.get_IBIangles import get_IBIangles
from plot_functions.plt_tools import (set_font_type, defaultPlotting,day_night_split)
from plot_functions.plt_functions import (plt_categorical_grid, plt_network_graphs)
from plot_functions.get_bout_correlation import get_cluster_phaseSpace
from plot_functions import simfish
import math
from plot_functions.plt_tools import round_half_up 
from plot_functions.get_index import get_index
from scipy.signal import savgol_filter
from scipy.special import exp10
import statistics as s
import networkx as nx
import random
import simpy
from networkx.algorithms.community import greedy_modularity_communities
from netgraph import Graph

In [ ]:
set_font_type()
# mpl.rc('figure', max_open_warning = 0)
# os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [ ]:
# Select data and create figure folder
pick_data = 'pclesion'
which_ztime = 'day'
compare_which = 'cond1' # condition for separation None for treat as whole
nCluster = 12
sort_by_feature = 'pitch_initial' # by which parameter to sort the clusters on the figure

root, FRAME_RATE = get_data_dir(pick_data)

folder_name = f'{pick_data} cluster_sim_by{compare_which}'
folder_dir = get_figure_dir(pick_data)
fig_dir = os.path.join(folder_dir, folder_name)

try:
    os.makedirs(fig_dir)
    print(f'fig folder created: {folder_name}')
except:
    print('Notes: re-writing old figures')

Notes: re-writing old figures


In [ ]:
all_around_peak_data, all_feature_cond, all_cond0, all_cond1, idxRANGE = get_aligned_bouts_wIBI(root, FRAME_RATE, ztime=which_ztime)
IBI_angles, cond0, cond1 = get_IBIangles(root, FRAME_RATE, ztime=which_ztime)

In [ ]:
all_feature_clustered = get_cluster_phaseSpace(all_around_peak_data, all_feature_cond, idxRANGE, nCluster)

In [ ]:
connected_bout_features = all_feature_clustered[all_feature_clustered['to_bout'].notna()]

to_cluster = connected_bout_features[['to_bout']].merge(all_feature_clustered[['cluster','bout_uid']], left_on='to_bout',right_on='bout_uid',how='left')

to_cluster.columns = ['to_bout', 'to_cluster', 'to_bout_uid']

connected_bout_features = connected_bout_features.merge(to_cluster[['to_bout','to_cluster']], on='to_bout', how='left')
connected_bout_features = connected_bout_features[connected_bout_features['to_cluster'].notna()]
connected_bout_features.reset_index(drop=True, inplace=True)
connected_bout_features = connected_bout_features.assign(
    to_cluster = connected_bout_features['to_cluster'].astype('int'),
)

In [ ]:
if bool(compare_which):
    for this_condition in connected_bout_features[compare_which].unique():
        this_cond_features = connected_bout_features.loc[connected_bout_features[compare_which]==this_condition]
        # plot_network_graphs(extracted_features=this_cond_features, cond_sep=this_condition, sort_by_feature='ydispl_swim', total_features=connected_bout_features)
        extracted_features=this_cond_features
        cond_sep=this_condition
        total_features=connected_bout_features
        print(this_condition)
        plt_network_graphs(connected_bout_features, 
                            fig_dir = fig_dir,
                            sort_by_feature = sort_by_feature,
                            cond_sep=this_condition, 
                            extracted_features=this_cond_features)
else:
    plt_network_graphs(connected_bout_features, 
                        fig_dir = fig_dir,
                        sort_by_feature = sort_by_feature)

1ctrl


TypeError: agg function failed [how->mean,dtype->object]

In [ ]:
import matplotlib as mpl
import os
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from plot_functions.plt_tools import set_font_type
import warnings
import networkx as nx



def plt_categorical_combined(
    data:pd.DataFrame, 
    x:str, 
    y:str, 
    units:str, 
    row:str=None, 
    col:str=None, 
    errorbar=None, 
    sharey=True, 
    sns_palette='colorblind', 
    markertype='_', 
    height=3, 
    aspect=0.8,
    related=False,
    size=6,
    alpha=0.5,
    overlay_func = sns.swarmplot,
    col_order=None,
    estimator=np.nanmean
    ):
    
    """build upon sns.catplot(), plots mean y_name vs x_name with individual repeats. Repeats are plotted as stripplot if number of repeats are different among groups defined by x_name, otherwise, repeats are connected.
    the errorbar arg requires Seaborn v0.12 to work.

    Args:
        data (pd.DataFrame): Dataframe with one value per "unit". Can be achieved using: data.groupby([x_name, gridcol, gridrow, units]).mean().reset_index()
        x_name (str): Column to plot on the X axis.
        y_name (str): Column to plot on the Y axis. 
        gridrow (str): Categorical variables that will determine the faceting of the grid.
        gridcol (str): Categorical variables that will determine the faceting of the grid.
        units (str): Units for plotting individual repeats. see sns.lineplot() units arg.
        errorbar (optional): Defines type of error bars to plot. see seaborn.catplot errorbar arg. Defaults to None.
        sharey (bool, optional): Whether to share Y axis ticks. Defaults to True.
        sns_palette (str, optional): Color palettes. Defaults to 'colorblind'.
        height (int, optional): Height of the graph in inches. Defaults to 4.
        aspect (float, optional): aspect ratio of the graph. Defaults to 0.8.
    """
    set_font_type()
    cat_cols = [x, units, row, col]
    cat_cols = [ele for ele in cat_cols if ele is not None]

    if_one_val_per_rep = data.groupby(cat_cols).size().max()
    if if_one_val_per_rep > 1:
        data = data.groupby(cat_cols)[y].mean().reset_index()
    else:
        data = data.sort_values(by=cat_cols).reset_index(drop=True)
            
    assert_repeats = len(set(data.groupby([x])[units].apply(lambda x: len(x.unique())).values))
    
    if not related:
        assert_repeats = 0

        
    g = sns.catplot(
        data = data,
        col = col,
        row = row,
        # hue = x,
        x = x,
        y = y,
        kind = 'point',
        sharey = sharey,
        palette = sns_palette,
        errorbar = errorbar,
        markers = [markertype]*len(set(data[x])),
        height = height,
        aspect = aspect,
        col_order=col_order,
        estimator=estimator
        )
    if assert_repeats == 1:
        g.map(sns.lineplot,x,y,
            estimator=None,
            units=units,
            data = data,
            sort=False,
            color='grey',
            alpha=alpha,
            zorder=0,
            )
    else:
        if overlay_func:
            g.map(overlay_func,x,y,
                data = data,
                color='grey',
                alpha=alpha,
                zorder=0,
                order=data[x].unique().sort(),
                s=size,
            )
    g.add_legend()
    sns.despine(offset=10, trim=False)
    return g


def plt_categorical_grid2(
    data:pd.DataFrame, 
    x_name:str, 
    y_name:str, 
    units:str, 
    gridrow:str=None, 
    gridcol:str=None, 
    errorbar=None, 
    sharey=True, 
    sns_palette='colorblind', 
    markertype='d', 
    height=3, 
    aspect=0.8,
    method='mean'
    ):
    
    """build upon sns.catplot(), plots mean y_name vs x_name with individual repeats. Repeats are plotted as stripplot if number of repeats are different among groups defined by x_name, otherwise, repeats are connected.
    the errorbar arg requires Seaborn v0.12 to work.

    Args:
        data (pd.DataFrame): Dataframe with one value per "unit". Can be achieved using: data.groupby([x_name, gridcol, gridrow, units]).mean().reset_index()
        x_name (str): Column to plot on the X axis.
        y_name (str): Column to plot on the Y axis. 
        gridrow (str): Categorical variables that will determine the faceting of the grid.
        gridcol (str): Categorical variables that will determine the faceting of the grid.
        units (str): Units for plotting individual repeats. see sns.lineplot() units arg.
        errorbar (optional): Defines type of error bars to plot. see seaborn.catplot errorbar arg. Defaults to None.
        sharey (bool, optional): Whether to share Y axis ticks. Defaults to True.
        sns_palette (str, optional): Color palettes. Defaults to 'colorblind'.
        height (int, optional): Height of the graph in inches. Defaults to 4.
        aspect (float, optional): aspect ratio of the graph. Defaults to 0.8.
    """
    set_font_type()
    cat_cols = [x_name, units, gridrow, gridcol]
    cat_cols = [ele for ele in cat_cols if ele is not None]

    if_one_val_per_rep = data.groupby(cat_cols).size().max()
    if if_one_val_per_rep > 1:
        if method == 'mean':
            data = data.groupby(cat_cols)[y_name].mean().reset_index()
        if method == 'median':
            data = data.groupby(cat_cols)[y_name].median().reset_index()
    else:
        data = data.sort_values(by=cat_cols).reset_index(drop=True)
    
    assert_repeats = []    
    assert_df = pd.DataFrame(data.groupby([x_name])[units].apply(lambda x: x.unique()).reset_index()[units].tolist())  
    for col in assert_df:
        assert_repeats.append(len(assert_df[col].unique()))
        
    g = sns.catplot(
        data = data,
        col = gridcol,
        row = gridrow,
        hue = x_name,
        x = x_name,
        y = y_name,
        kind = 'point',
        sharey = sharey,
        palette = sns_palette,
        errorbar = errorbar,
        markers = [markertype]*len(set(data[x_name])),
        height = height,
        aspect = aspect,

        )
    if np.max(assert_repeats) == 1:
        g.map(sns.lineplot,x_name,y_name,
            estimator=None,
            units=units,
            data = data,
            sort=False,
            color='grey',
            alpha=0.2,
            zorder=0,
            )
    else:
        g.map(sns.stripplot,x_name,y_name,
            data = data,
            color='lightgrey',
            zorder=0,
            order=data[x_name].unique().sort(),
            )
    g.add_legend()
    sns.despine(offset=10, trim=False)
    return g

def plt_categorical_grid(
    data:pd.DataFrame, 
    x_name:str, 
    y_name:str, 
    gridrow:None, 
    gridcol:None, 
    units:str, 
    errorbar=None, 
    sharey=True, 
    sns_palette='colorblind', 
    markertype='d', 
    height=4, 
    aspect=0.8,
    ):
    
    """build upon sns.catplot(), plots mean y_name vs x_name with individual repeats. Repeats are plotted as stripplot if number of repeats are different among groups defined by x_name, otherwise, repeats are connected.
    the errorbar arg requires Seaborn v0.12 to work.

    Args:
        data (pd.DataFrame): Dataframe with one value per "unit". Can be achieved using: data.groupby([x_name, gridcol, gridrow, units]).mean().reset_index()
        x_name (str): Column to plot on the X axis.
        y_name (str): Column to plot on the Y axis. 
        gridrow (str): Categorical variables that will determine the faceting of the grid.
        gridcol (str): Categorical variables that will determine the faceting of the grid.
        units (str): Units for plotting individual repeats. see sns.lineplot() units arg.
        errorbar (optional): Defines type of error bars to plot. see seaborn.catplot errorbar arg. Defaults to None.
        sharey (bool, optional): Whether to share Y axis ticks. Defaults to True.
        sns_palette (str, optional): Color palettes. Defaults to 'colorblind'.
        height (int, optional): Height of the graph in inches. Defaults to 4.
        aspect (float, optional): aspect ratio of the graph. Defaults to 0.8.
    """
    set_font_type()
    data.sort_values(by=x_name,inplace=True)
    
    assert_repeats = data.groupby([x_name,units]).size().min()
    
    g = sns.catplot(
        data = data,
        col = gridcol,
        row = gridrow,
        hue = x_name,
        x = x_name,
        y = y_name,
        kind = 'point',
        sharey = sharey,
        palette = sns_palette,
        errorbar = errorbar,
        markers = [markertype]*len(set(data[x_name])),
        height = height,
        aspect = aspect,
        )
    if assert_repeats > 1:
        g.map(sns.lineplot,x_name,y_name,
            estimator=None,
            units=units,
            data = data,
            sort=False,
            color='grey',
            alpha=0.2,
            zorder=0,
            )
    else:
        g.map(sns.stripplot,x_name,y_name,
            data = data,
            color='lightgrey',
            zorder=0,
            order=data[x_name].unique().sort(),
            )
    g.add_legend()
    sns.despine(offset=10, trim=False)
    return g

def plt_network_graphs(total_features, fig_dir, sort_by_feature=[], cond_sep=None, extracted_features:pd.DataFrame=pd.DataFrame()):
# #     % norm by outbound
    if extracted_features.empty:
        extracted_features = total_features
        
    G_master = nx.from_pandas_edgelist(total_features.groupby(['cluster','to_cluster']).size().reset_index(), 'cluster', 'to_cluster', create_using=nx.DiGraph()) 
    pos = nx.circular_layout(G_master)
    nCluster = len(list(G_master))

    if sort_by_feature:
        if type(sort_by_feature) == str:
            sort_by_feature = [sort_by_feature]
            
    for sel_feature_to_sort in sort_by_feature:
        # % colored by outbound and reorder
        sorted_cluster = total_features.groupby('cluster').mean(numeric_only=True).sort_values(by=sel_feature_to_sort).reset_index().reset_index()
        cluster_seq = sorted_cluster['cluster']

        connected_bouts = extracted_features.loc[:,['cluster','to_cluster','expNum','cond0','cond1',sel_feature_to_sort]]

        graph_df = connected_bouts.groupby(['cluster','to_cluster']).size().reset_index()
        graph_df.columns = ['from_cluster','to_cluster','weight']

        # normalize by the number of bouts per cluster
        graph_df = graph_df.merge(connected_bouts.groupby('cluster').size().to_frame(name='total_bouts').reset_index(),left_on='from_cluster', right_on='cluster')
        graph_df = graph_df.merge(sorted_cluster[['cluster',sel_feature_to_sort]], left_on='from_cluster', right_on='cluster')
        graph_df = graph_df.assign(
            weight_norm = graph_df['weight']/graph_df['total_bouts']
        )

        # pos = nx.circular_layout(G)

        pos_alt = {}
        for i, key in enumerate(cluster_seq):
            pos_alt[key] = pos[i]

        H = nx.from_pandas_edgelist(graph_df, 'from_cluster', 'to_cluster', ['weight_norm',sel_feature_to_sort], create_using=nx.DiGraph()) 
        G = nx.DiGraph()
        G.add_nodes_from(sorted(H.nodes(data=True)))
        G.add_edges_from(H.edges(data=True))
        # pos = nx.circular_layout(G)
        # labels = nx.get_edge_attributes(G,'weight_norm')

        edges = G.edges()
        weights = np.array([G[u][v]['weight_norm'] for u,v in edges])
        c_feature = np.array([G[u][v][sel_feature_to_sort] for u,v in edges])

        weights_adj = weights/weights.max()*4
        alpha = np.log((1+weights/np.max(weights))*(np.e/2))
        # c = sns.color_palette()
        c = sns.diverging_palette(250, 30, l=65, center="dark", as_cmap=True)
        c_list = sns.diverging_palette(250, 30, l=65, center="dark", n=nCluster)
        
        fig, (ax1) = plt.subplots(1, figsize=(8,5))
        nx.draw_networkx_edges(G,pos=pos_alt,
                            alpha = (alpha-alpha.min())/(1-alpha.min()),
                            width = weights_adj,
                            connectionstyle='arc3, rad=0.1',
                            edge_color = c_feature,
                            edge_cmap = c,
                            ax=ax1,
                            #    arrowstyle='-|>' 
                            )

            
        nx.draw_networkx_nodes(
            G,
            pos=pos_alt,
            ax=ax1,
            node_color=[c_list[i] for i in cluster_seq.sort_values().index]
        )
        nx.draw_networkx_labels(
            G,
            pos=pos_alt,
            font_color='w',
            ax=ax1
        )

        edges2 = nx.draw_networkx_edges(G,pos=pos_alt,
                            alpha = 1,
                            connectionstyle='arc3, rad=0.1',
                            edge_color = c_feature,
                            width=0,
                            edge_cmap = c,
                            ax=ax1,
                            arrows=False
        )
        cbar = plt.colorbar(edges2, ax=ax1)
        cbar.ax.set_ylabel(sel_feature_to_sort, rotation=270)

        plt.savefig(f"{fig_dir}/graph_c{nCluster}_outbound_by{sel_feature_to_sort}_{cond_sep}.pdf",format='PDF')
        
        ############# by total

        H = nx.from_pandas_edgelist(graph_df, 'from_cluster', 'to_cluster', ['weight',sel_feature_to_sort], create_using=nx.DiGraph()) 
        G = nx.DiGraph()
        G.add_nodes_from(sorted(H.nodes(data=True)))
        G.add_edges_from(H.edges(data=True))
        
        labels = nx.get_edge_attributes(G,'weight')

        edges = G.edges()
        weights = np.array([G[u][v]['weight'] for u,v in edges])
        c_feature = np.array([G[u][v][sel_feature_to_sort] for u,v in edges])

        weights_adj = weights/weights.max()*4
        alpha = np.log((1+weights/np.max(weights))*(np.e/2))
        # c = sns.color_palette("flare", as_cmap=True)
        # c = sns.light_palette("black", as_cmap=True)
        c = sns.diverging_palette(250, 30, l=65, center="dark", as_cmap=True)
        c_list = sns.diverging_palette(250, 30, l=65, center="dark", n=nCluster)

        fig, (ax1) = plt.subplots(1, figsize=(8,5))
        nx.draw_networkx_edges(G,pos=pos_alt,
                            alpha = alpha,
                            width = weights_adj,
                            connectionstyle='arc3, rad=0.1',
                            edge_color = c_feature,
                            edge_cmap = c,
                            #    arrowstyle='-|>' 
                            )

            
        nx.draw_networkx_nodes(
            G,
            pos=pos_alt,
            ax=ax1,
            node_color=[c_list[i] for i in cluster_seq.sort_values().index]
        )
        nx.draw_networkx_labels(
            G,
            pos=pos_alt,
            font_color='w',
            ax=ax1
        )
        edges2 = nx.draw_networkx_edges(G,pos=pos_alt,
                    alpha = 1,
                    connectionstyle='arc3, rad=0.1',
                    edge_color = c_feature,
                    width=0,
                    edge_cmap = c,
                    ax=ax1,
                    arrows=False
        )
        cbar = plt.colorbar(edges2, ax=ax1)
        cbar.ax.set_ylabel(sel_feature_to_sort, rotation=270)

        plt.savefig(f"{fig_dir}/graph_c{nCluster}_total_by{sel_feature_to_sort}_{cond_sep}.pdf",format='PDF')

    # pos = nx.circular_layout(G)
#     # labels = nx.get_edge_attributes(G,'weight_norm')

#     edges = G.edges()
#     weights = np.array([G[u][v]['weight_norm'] for u,v in edges])

#     weights_adj = weights/weights.max()*3
#     alpha = np.log((1+weights/np.max(weights))*(np.e/2))
#     # c = sns.color_palette("flare", as_cmap=True)
#     c = sns.light_palette("black", as_cmap=True)


#     fig = plt.figure(figsize=(7,5))
#     arrows = nx.draw_networkx_edges(G,pos=pos,
#                         alpha = alpha,
#                         width = weights_adj,
#                         connectionstyle='arc3, rad=0.1',
#                         edge_color = weights,
#                         edge_cmap = c,
#                         #    arrowstyle='-|>' 
#                         )

        
#     nx.draw_networkx_nodes(
#         G,
#         pos=pos,
#     )
#     nx.draw_networkx_labels(
#         G,
#         pos=pos,
#     )
#     plt.savefig(f"{fig_dir}/graph_c{nCluster}_normToCluster_{cond_sep}.pdf",format='PDF')

#     ############################
#     # % total

#     G = nx.from_pandas_edgelist(graph_df, 'from_cluster', 'to_cluster', 'weight', create_using=nx.DiGraph()) 
#     # pos = nx.circular_layout(G)
#     labels = nx.get_edge_attributes(G,'weight')

#     edges = G.edges()
#     weights = np.array([G[u][v]['weight'] for u,v in edges])

#     weights_adj = weights/weights.max()*3
#     alpha = np.log((1+weights/np.max(weights))*(np.e/2))
#     # c = sns.color_palette("flare", as_cmap=True)
#     c = sns.light_palette("black", as_cmap=True)


#     fig = plt.figure(figsize=(7,5))
#     arrows = nx.draw_networkx_edges(G,pos=pos,
#                         alpha = 1,
#                         width = weights_adj,
#                         connectionstyle='arc3, rad=0.1',
#                         edge_color = weights,
#                         edge_cmap = c,
#                         #    arrowstyle='-|>' 
#                         )

        
#     nx.draw_networkx_nodes(
#         G,
#         pos=pos,
#     )
#     nx.draw_networkx_labels(
#         G,
#         pos=pos,
#     )

#     plt.savefig(f"{fig_dir}/graph_c{nCluster}_totalBouts_{cond_sep}.pdf",format='PDF')
    
    ############################